# AI002 Final Pipeline: Dự báo Giá Cà phê & Khuyến nghị Canh tác
**Dự án:** DT10
**Data:** Dữ liệu thực tế các vùng trồng cà phê (2020-2026)

Pipeline này thể hiện sự áp dụng của **5 Trụ cột AI Bền vững (Sustainable AI)**:
1. **Reliability:** Đánh giá RMSE/MAE với split thời gian: train 2020-2024, test 2025, 2026 dùng cho demo/inference/audit.
2. **Bias:** Nhận diện và đo lường sai số giữa các khu vực địa lý khác nhau.
3. **Robustness:** Xử lý missing values bằng TDD validation và Imputation.
4. **Transparency:** Mô hình Tree-based (Random Forest) với Feature Importance.
5. **Social Impact:** Từ kết quả dự báo, đưa ra lời khuyên canh tác cụ thể cho nông dân.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


## 1. Nạp dữ liệu và Kiểm định (Robustness & TDD)
Sử dụng bộ dữ liệu thực tế đã qua tổng hợp `coffee_environment_all_areas_monthly_2020_2026.csv`.


In [ ]:
# Load data
df = pd.read_csv('../data/processed/monthly/coffee_environment_all_areas_monthly_2020_2026.csv')
df['period_start'] = pd.to_datetime(df['period_start'])
df = df.sort_values(by=['area', 'period_start']).reset_index(drop=True)

# TDD: Data Validation
assert not df.empty, "Dữ liệu rỗng!"
assert 'avg_price_vnd_per_kg' in df.columns, "Thiếu cột target!"

# Xử lý Missing Values (Robustness)
# Trong dữ liệu thật, có một số tháng thiếu quan sát giá (interpolated_area).
# Ta sẽ fillna cho các feature thời tiết nếu có.
weather_cols = ['avg_temperature_c', 'avg_humidity_percent', 'total_rainfall_mm']
df[weather_cols] = df.groupby('area')[weather_cols].transform(lambda x: x.ffill().bfill())

print(f"Dataset shape: {df.shape}")
df.head(3)


## 2. Exploratory Data Analysis & Bias Check
Kiểm tra xem dữ liệu có thiên lệch (Bias) giữa các vùng miền hay không.

In [ ]:
# Plot giá theo vùng (Bias Check)
plt.figure(figsize=(14, 6))
sns.boxplot(x='area', y='avg_price_vnd_per_kg', data=df)
plt.title("Hình 1. Phân bố giá cà phê theo khu vực")
plt.xlabel("Khu vực trong dataset Tây Nguyên")
plt.ylabel("Giá cà phê trung bình theo tháng (VND/kg)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(Markdown("""
**Chú thích Hình 1 — Phân bố giá cà phê theo khu vực**

- **Trục X** là từng khu vực trồng cà phê trong dataset Tây Nguyên; **trục Y** là giá cà phê trung bình theo tháng, đơn vị VND/kg.
- Đường giữa hộp là mức giá trung vị của khu vực; hộp càng cao nghĩa là giá khu vực đó biến động càng mạnh qua các tháng. Các điểm nằm xa hộp là tháng có giá bất thường.
- Hình này dùng cho phần **Bias**: nếu một khu vực có phân bố quá khác phần còn lại, mô hình có nguy cơ dự báo tốt cho vùng đó nhưng kém ổn định khi suy rộng sang vùng khác. Khi viết báo cáo, kết luận nên là mô hình phù hợp nhất cho các vùng có phân bố tương tự dữ liệu huấn luyện, không đại diện cho mọi vùng trồng cà phê ở Việt Nam.
"""))


## 3. Feature Engineering (Reliability)
Tạo độ trễ (Lags) vì giá cà phê tháng này phụ thuộc rất lớn vào giá tháng trước và thời tiết tháng trước.

In [ ]:
# Tạo Lag features theo từng khu vực
df['price_lag_1'] = df.groupby('area')['avg_price_vnd_per_kg'].shift(1)
df['temp_lag_1'] = df.groupby('area')['avg_temperature_c'].shift(1)
df['rain_lag_1'] = df.groupby('area')['total_rainfall_mm'].shift(1)

# Drop NaN do phép shift tạo ra
df_model = df.dropna().copy()

# Encoding biến phân loại (Area)
df_model = pd.get_dummies(df_model, columns=['area'], drop_first=True)

# Các features sử dụng
features = ['price_lag_1', 'temp_lag_1', 'rain_lag_1', 'avg_temperature_c', 'total_rainfall_mm', 'avg_humidity_percent']
area_cols = [c for c in df_model.columns if c.startswith('area_')]
features.extend(area_cols)

target = 'avg_price_vnd_per_kg'
print(f"Tổng số features: {len(features)}")


## 4. Huấn luyện Mô hình (Transparency)
- Sử dụng **Temporal Split**: Train 2020-2024, Test 2025; dữ liệu 2026 chỉ dùng cho demo/inference/audit.
- Model: **Random Forest** (Không dùng Blackbox, đảm bảo tính minh bạch giải thích được).


In [ ]:
# Time Series Split
train_mask = df_model['period_start'].dt.year <= 2024
test_mask = df_model['period_start'].dt.year == 2025
demo_mask = df_model['period_start'].dt.year >= 2026

X_train, y_train = df_model[train_mask][features], df_model[train_mask][target]
X_test, y_test = df_model[test_mask][features], df_model[test_mask][target]
demo_rows = df_model[demo_mask]

# Train Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"=== Đánh giá mô hình (Reliability) ===")
print(f"MAE: {mae:,.0f} VND")
print(f"RMSE: {rmse:,.0f} VND")
print(f"Số dòng train: {len(X_train):,}")
print(f"Số dòng test 2025: {len(X_test):,}")
print(f"Số dòng 2026 giữ lại cho demo/inference/audit: {len(demo_rows):,}")

# Plot dự báo vs thực tế theo tháng để người đọc thấy xu hướng 2025 rõ hơn.
test_result = df_model.loc[test_mask, ['period_start']].copy()
test_result['actual_price_vnd'] = y_test.to_numpy()
test_result['predicted_price_vnd'] = y_pred
monthly_result = (
    test_result.groupby('period_start', as_index=False)[['actual_price_vnd', 'predicted_price_vnd']]
    .mean()
    .sort_values('period_start')
)

plt.figure(figsize=(12, 5))
plt.plot(monthly_result['period_start'], monthly_result['actual_price_vnd'], label='Giá thực tế 2025', marker='o')
plt.plot(monthly_result['period_start'], monthly_result['predicted_price_vnd'], label='Giá dự báo 2025', marker='x', linestyle='--')
plt.title("Hình 2. Giá thực tế và dự báo trung bình theo tháng trên tập test 2025")
plt.xlabel("Tháng trong năm 2025")
plt.ylabel("Giá cà phê trung bình (VND/kg)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

display(Markdown("""
**Chú thích Hình 2 — Giá thực tế và dự báo trung bình theo tháng trên tập test 2025**

- **Trục X** là các tháng trong năm 2025; **trục Y** là giá cà phê trung bình VND/kg sau khi gom các khu vực trong tập test. Đường liền là giá thực tế, đường nét đứt là giá mô hình dự báo.
- Hình này dùng cho phần **Reliability**: hai đường càng gần nhau thì mô hình càng bám sát diễn biến giá năm 2025. Khoảng cách giữa hai đường ở từng tháng là sai số trực quan, bổ sung cho MAE/RMSE in ngay phía trên cell.
- Cách kết luận nên cụ thể: nếu đường dự báo đi cùng chiều với đường thực tế nhưng thấp/cao hơn ở một số tháng, mô hình bắt được xu hướng chung nhưng còn sai số khi thị trường biến động mạnh. Dữ liệu 2026 không nằm trong đường này vì chỉ dùng cho demo/inference/audit, không dùng để tính điểm test.
"""))


## 5. Minh bạch & Tác động Xã hội (Transparency & Social Impact)
Xuất Feature Importance để giải thích mô hình và đưa ra khuyến nghị canh tác.

In [ ]:
# Feature Importance (Transparency)
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp.head(10))
plt.title("Hình 3. Top 10 đặc trưng ảnh hưởng đến dự báo giá")
plt.xlabel("Mức độ quan trọng trong Random Forest")
plt.ylabel("Đặc trưng đầu vào")
plt.tight_layout()
plt.show()

display(Markdown("""
**Chú thích Hình 3 — Top 10 đặc trưng ảnh hưởng đến dự báo giá**

- **Trục Y** là tên đặc trưng đầu vào; **trục X** là mức độ quan trọng do Random Forest tính. Thanh càng dài nghĩa là đặc trưng đó được mô hình dùng nhiều hơn khi chia cây và giảm lỗi dự báo.
- Hình này dùng cho phần **Transparency**: người đọc có thể biết mô hình dựa nhiều vào yếu tố nào, ví dụ giá tháng trước (`price_lag_1`), thời tiết tháng trước (`temp_lag_1`, `rain_lag_1`), thời tiết hiện tại, hoặc biến khu vực dạng one-hot.
- Cách kết luận phải gắn với thanh cao nhất trong hình sau khi chạy notebook. Nếu `price_lag_1` đứng đầu, kết luận là baseline chủ yếu học quán tính giá quá khứ; nếu biến thời tiết/khu vực đứng cao, có thể nói thời tiết/khu vực đóng góp rõ hơn. Không nên viết chung chung rằng mô hình “hiểu đầy đủ nông học”; hình chỉ chứng minh mức đóng góp tương đối của các feature đã đưa vào notebook.
"""))

# Đọc Rules Canh Tác (Social Impact)
try:
    with open('../data/processed/farming_advisory_rules.json', 'r', encoding='utf-8') as f:
        rules = json.load(f)
    print("=== Trích xuất Lời khuyên Canh tác (Social Impact) ===")
    print("Rule 1:", rules.get('drought_warning', {}).get('action', 'Tăng cường tưới tiêu'))
    print("Mô hình dự báo có thể kết hợp với bộ rule này để nhắn tin SMS tự động cho nông dân khi có thời tiết bất thường.")
except FileNotFoundError:
    print("Không tìm thấy file farming_advisory_rules.json, vui lòng kiểm tra thư mục data/processed/")
